###  Implement a Deep Q-Network (DQN) using a deep neural network library (e.g., TensorFlow or PyTorch) and train it on a simple environment like CartPole or Mountain Car.

In [3]:
# Import libraries
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

In [20]:
# Create environment (CartPole)
env = gym.make("CartPole-v1")

# State = 4 values (cart position, velocity, pole angle, etc.)
state_dim = env.observation_space.shape[0]

# Actions = 2 (move left or right)
action_dim = env.action_space.n

print("State:", state_dim, "Actions:", action_dim)

State: 4 Actions: 2


## Q-Network (Core Idea of DQN)

In [21]:
# This network learns Q(s,a)
# Q-value = "How good is this action in this state?"

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),  # input = state
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)  # output = Q-values for all actions
        )

    def forward(self, x):
        return self.net(x)

## Replay Buffer (Memory)

In [22]:
# Stores past experiences to make learning stable

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        # Save one experience
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size=64):
        # Random sample (breaks correlation)
        batch = random.sample(self.buffer, batch_size)

        s, a, r, ns, d = zip(*batch)

        return (
            torch.tensor(s, dtype=torch.float32),
            torch.tensor(a, dtype=torch.long),      # actions must be long
            torch.tensor(r, dtype=torch.float32),
            torch.tensor(ns, dtype=torch.float32),
            torch.tensor(d, dtype=torch.float32)
        )

## Initialize Model & Hyperparameters

In [23]:
model = DQN(state_dim, action_dim)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
buffer = ReplayBuffer()

# Discount factor → importance of future rewards
gamma = 0.99

# Exploration vs Exploitation
epsilon = 1.0        # start exploring
epsilon_decay = 0.995
epsilon_min = 0.05

## Action Selection (ε-greedy)

In [24]:
# With probability epsilon → random action (exploration)
# Otherwise → best action from model (exploitation)

def select_action(state):
    # Exploration
    if random.random() < epsilon:
        return env.action_space.sample(), "Explore"

    # Exploitation
    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    q_values = model(state_tensor)[0]

    action = torch.argmax(q_values).item()
    return action, "Exploit"

## Training Step (Bellman Update)

In [25]:
# Learn using Bellman Equation:
# Q(s,a) = r + gamma * max Q(s',a')

def train():
    if len(buffer.buffer) < 100:
        return

    states, actions, rewards, next_states, dones = buffer.sample()

    # Current Q values for selected actions
    q_values = model(states)
    q_value = q_values.gather(1, actions.unsqueeze(1)).squeeze()

    # Best Q value for next state
    next_q_values = model(next_states).max(1)[0]

    # Target Q value
    target = rewards + gamma * next_q_values * (1 - dones)

    # Loss = difference between predicted and target
    loss = ((q_value - target.detach()) ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [26]:
episodes = 10  # keep small for understanding

for ep in range(episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False
    step = 0

    print(f"\n===== EPISODE {ep+1} =====")

    while not done:
        step += 1

        # Get action
        action, mode = select_action(state)

        # Q-values (for explanation only)
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        q_values = model(state_tensor)[0]

        # Take step
        next_state, reward, done, _, _ = env.step(action)

        # Store experience
        buffer.add(state, action, reward, next_state, done)

        # Train
        train()

        # ---- PRINT EXPLANATION ----
        print(f"Step {step}")
        print(f"State: {state}")
        print(f"Q-values: {q_values.detach()}")
        print(f"Action: {action} ({mode})")
        print(f"Reward: {reward}")
        print("-" * 40)

        state = next_state
        total_reward += reward

    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    print(f"Episode {ep+1} Total Reward: {total_reward}")


===== EPISODE 1 =====
Step 1
State: [-0.04894079 -0.02319565  0.00150982 -0.0410437 ]
Q-values: tensor([0.0205, 0.1087])
Action: 1 (Explore)
Reward: 1.0
----------------------------------------
Step 2
State: [-0.0494047   0.17190461  0.00068895 -0.33324987]
Q-values: tensor([0.0269, 0.1442])
Action: 1 (Explore)
Reward: 1.0
----------------------------------------
Step 3
State: [-0.04596661  0.36701676 -0.00597605 -0.62571543]
Q-values: tensor([0.0185, 0.1699])
Action: 1 (Explore)
Reward: 1.0
----------------------------------------
Step 4
State: [-0.03862627  0.5622216  -0.01849036 -0.9202745 ]
Q-values: tensor([0.0150, 0.1963])
Action: 1 (Explore)
Reward: 1.0
----------------------------------------
Step 5
State: [-0.02738184  0.7575885  -0.03689585 -1.2187105 ]
Q-values: tensor([0.0204, 0.2209])
Action: 0 (Explore)
Reward: 1.0
----------------------------------------
Step 6
State: [-0.01223007  0.56296116 -0.06127006 -0.93781316]
Q-values: tensor([0.0173, 0.2014])
Action: 1 (Explore

### 🔍 Output Explanation (Short)

* **State** → Current situation of the environment
  *(cart position, velocity, pole angle, pole velocity)*

* **Q-values** → Model’s prediction of future reward for each action

  * First value → Action 0 (move left)
  * Second value → Action 1 (move right)
  * Higher value = better action

* **Action** → Decision taken by agent

  * **Explore** → random action (to learn)
  * **Exploit** → best action based on Q-values

* **Reward** → Immediate reward from environment
  *(CartPole gives +1 for every step survived)*

* **Episode Reward (36)** → Total steps the pole stayed balanced

👉 As training improves:

* Q-values become more accurate
* Agent exploits more
* Total reward increases (goal ≈ 200)
